# Fase 4 — Jurisprudencia CSJN (SAIJ)

Baja los **fallos de la Corte Suprema de Justicia de la Nación publicados en SAIJ desde 2020**
(alcance acordado por el equipo el 2026-07-08), los segmenta, les genera embeddings con el mismo
pipeline de la Fase 2 y construye el vínculo ley↔fallo por similitud semántica.

Fuente: API JSON de SAIJ (misma familia de endpoints que produjo `saij_texts.csv`). El texto
completo de cada fallo viene como PDF; se extrae con `pypdf`. Detalle de la API en
`src/lexar/saij.py`.

Outputs:
- `data/jurisprudencia_csjn/fallos_csjn.parquet` — un fallo por fila (gitignored).
- `outputs/jurisprudencia/case_fragments.parquet` + `case_embeddings.npy` — fragmentos únicos y
  su matriz de embeddings alineada 1:1.
- `outputs/jurisprudencia/law_case_links.parquet` — pares (norma, fallo) por similitud.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

# El paquete src/lexar vive en la raiz del repo; soporta lanzar Jupyter desde la raiz o notebooks/.
for _base in [Path.cwd(), *Path.cwd().parents]:
    if (_base / "src" / "lexar").exists():
        sys.path.insert(0, str(_base / "src"))
        break

import json

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 220)

from lexar import config
from lexar.saij import scrape_fallos_csjn, consolidate_fallos, build_case_fragments
from lexar.embeddings import embed_corpus_checkpointed, consolidate_embeddings
from lexar.links import build_law_case_links

# None = corrida completa. Bajar (ej. 20) para un smoke test barato antes de una corrida real.
MAX_FALLOS = None
SINCE = "2020-01-01"

PARTS_DIR = config.JURIS_DATA_DIR / "parts"
EMBED_CHECKPOINT_DIR = config.JURIS_OUTPUT_DIR / "embeddings"

config.JURIS_DATA_DIR, config.JURIS_OUTPUT_DIR

## 4.1 Scraping

Reanudable: los fallos ya bajados en checkpoints previos se saltean, así que re-ejecutar esta
celda es barato. El corte por fecha usa la `fecha` real de cada documento (la búsqueda viene en
orden fecha DESC y se corta tras una racha de fallos anteriores a `SINCE`).

In [ ]:
scrape_fallos_csjn(PARTS_DIR, since=SINCE, max_fallos=MAX_FALLOS)
fallos = consolidate_fallos(PARTS_DIR, config.FALLOS_PATH, since=SINCE)

print(fallos["fecha"].str[:4].value_counts().sort_index())
print(fallos["tipo_fallo"].value_counts())
fallos[["case_id", "fecha", "tipo_fallo", "actor", "demandado", "sobre"]].head(5)

## 4.2 Segmentación

Los fallos no tienen estructura `ARTICULO n`, así que se usa `chunk_text()` directo (el mismo
fallback que la Fase 1 aplica a leyes sin artículos). Se deduplica por `content_hash` — la
tabla guardada queda alineada 1:1 con la matriz de embeddings.

In [ ]:
case_fragments_all = build_case_fragments(fallos)
print(f"Fragmentos generados: {len(case_fragments_all):,}")

case_fragments = case_fragments_all.drop_duplicates("content_hash").reset_index(drop=True)
print(f"Fragmentos únicos por content_hash: {len(case_fragments):,}")
print(case_fragments["text_len"].describe().round(1))
case_fragments.head(3)

## 4.3 Embeddings

Mismo pipeline de la Fase 2: `gemini-embedding-001` (768d, `SEMANTIC_SIMILARITY` — mismo espacio
que el corpus de leyes), batches concurrentes con rate limiting adaptativo y checkpoints
reanudables por `content_hash`.

In [ ]:
failed = embed_corpus_checkpointed(case_fragments, EMBED_CHECKPOINT_DIR)
assert failed == 0, "Quedaron batches pendientes; re-ejecutar esta celda para reintentarlos."

case_embeddings = consolidate_embeddings(case_fragments, EMBED_CHECKPOINT_DIR)
np.save(config.CASE_EMBEDDINGS_NPY_PATH, case_embeddings)
case_fragments.to_parquet(config.CASE_FRAGMENTS_PATH, index=False)
print(f"Matriz de embeddings: {case_embeddings.shape}")

## 4.4 Vínculo ley↔fallo

Cada fragmento de fallo se busca contra el índice FAISS de los 112k fragmentos de leyes
(Fase 2) y se agrega a nivel `(document_id, case_id)`. Sin texto en la tabla bulk — misma
regla de memoria que `analysis_candidates` (ver CLAUDE.md).

In [ ]:
law_case_links = build_law_case_links(case_fragments, case_embeddings)
law_case_links.head(10)

In [ ]:
# Sanity: los vínculos más fuertes, con título de la norma y carátula del fallo.
documents = pd.read_csv(config.DOCUMENTS_PATH, dtype=str, keep_default_na=False)
titles = documents.set_index("document_id")["titulo_resumido"]
caratulas = fallos.set_index("case_id").apply(lambda f: f"{f['actor']} c/ {f['demandado']} s/ {f['sobre']}", axis=1)

preview = law_case_links.head(15).copy()
preview["norma"] = preview["document_id"].map(titles)
preview["fallo"] = preview["case_id"].map(caratulas).str[:110]
preview[["max_similarity", "n_fragment_pairs", "norma", "fallo"]]